# Настройка OSMnx


В разделе [«Экспорт данных из OSM»](spData_4.ipynb) мы научились выгружать объекты из **OpenStreetMap** с помощью библиотеки **OSMnx**. На практике у многих этот шаг не срабатывает: запрос «висит», обрывается по таймауту или возвращает ошибку сервера.

Дело в том, что OSMnx сама **не хранит** данные. Она лишь отправляет запросы к внешним публичным сервисам, а они бывают перегружены, временно недоступны или ограничивают частоту обращений.

> В этом разделе мы разберёмся, какие сервисы использует OSMnx, почему данные могут не загружаться, и научимся менять провайдера и другие настройки, чтобы выгрузка работала стабильно.


## 0. Импорт библиотек


In [1]:
import osmnx as ox

Проверим версию библиотеки — настройки, описанные ниже, актуальны для **OSMnx 2.x**:


In [2]:
ox.__version__

'2.0.7'

## 1. Почему данные не загружаются


OSMnx работает с двумя внешними сервисами, и каждый отвечает за свою задачу:

- **Nominatim** — геокодер. Используется, когда мы задаём территорию по названию: `ox.geocode_to_gdf()`, `ox.geocode()`, а также внутри `ox.features_from_place()`.
- **Overpass API** — сервер выгрузки объектов. Используется во всех функциях, которые скачивают сами данные: `ox.features_from_place()`, `ox.features_from_polygon()`, `ox.features_from_bbox()`, а также при загрузке уличной сети (`ox.graph_from_*`).

Это **бесплатные публичные серверы** с ограниченными ресурсами. Типичные причины, по которым загрузка не проходит:

1. `Timeout` / `ReadTimeout` -- Не хватило времени на ответ (большой запрос или медленный сервер)
2. Ошибки `429 Too Many Requests`, `504 Gateway Timeout` -- Сервер ограничивает частоту запросов или временно недоступен
3. `ConnectionError` -- Сервер недоступен из вашей сети / региона (тут можно попробовать VPN)


## 2. Где живут настройки: `ox.settings`


Все параметры OSMnx хранятся в объекте `ox.settings`. Чтобы изменить настройку, достаточно присвоить ей новое значение.

> **Важно:** настройки нужно задавать **до** вызова функций загрузки данных. OSMnx читает их в момент запроса, поэтому менять параметры имеет смысл в самом начале ноутбука, сразу после импорта.

Посмотреть текущее значение любой настройки можно, просто обратившись к ней:


In [3]:
# Текущий сервер Overpass
ox.settings.overpass_url

'https://overpass-api.de/api'

In [4]:
# Текущий геокодер Nominatim
ox.settings.nominatim_url

'https://nominatim.openstreetmap.org/'

## 3. Смена сервера Overpass


Самая частая причина проблем — перегруженный сервер Overpass, который стоит по умолчанию (`overpass-api.de`, Германия). Решение — переключиться на **зеркало** (mirror): другой сервер с теми же данными OSM.

Адрес сервера задаётся настройкой `ox.settings.overpass_url`. Указывается **базовый адрес** (без `/interpreter` на конце — OSMnx добавит его сам).

Несколько публичных зеркал, которые можно попробовать:

```python
# По умолчанию (Германия)
ox.settings.overpass_url = "https://overpass-api.de/api"

# Быстрое зеркало kumi.systems (Германия)
ox.settings.overpass_url = "https://overpass.kumi.systems/api"

# Зеркало private.coffee (Австрия)
ox.settings.overpass_url = "https://overpass.private.coffee/api"

# Зеркало VK / Mail.ru (часто стабильно работает из России)
ox.settings.overpass_url = "https://maps.mail.ru/osm/tools/overpass/api"
```

> Если одно зеркало не отвечает — пробуйте следующее. Сервера периодически меняются: актуальный список публичных серверов Overpass есть в [вики OpenStreetMap](https://wiki.openstreetmap.org/wiki/Overpass_API#Public_Overpass_API_instances).

Зададим зеркало, которое будем использовать:


In [ ]:
ox.settings.overpass_url = "https://maps.mail.ru/osm/tools/overpass/api"

## 4. Увеличение таймаута


Таймаут — это время, которое OSMnx ждёт ответа от сервера, прежде чем прервать запрос с ошибкой. По умолчанию это **180 секунд**.

Если территория большая или соединение медленное, ответ может просто не успевать прийти. В этом случае таймаут стоит увеличить:


In [ ]:
ox.settings.requests_timeout = 600  # секунд

> Большой таймаут не ускоряет загрузку – он лишь даёт серверу больше времени. Если запрос всё равно обрывается, проблема, скорее всего, в самом сервере (см. раздел 3) или в слишком большом объёме данных (попробуйте уменьшить территорию или сузить теги).


## 5. Кеширование запросов


OSMnx по умолчанию **кеширует** ответы серверов: результат каждого запроса сохраняется в локальную папку, и при повторном запуске того же запроса данные берутся уже с диска, а не из сети.

За это отвечают настройки:

- `ox.settings.use_cache` — включён ли кеш (по умолчанию `True`);
- `ox.settings.cache_folder` — папка для кеша (по умолчанию `./cache`).

> Не выключайте кеш без необходимости. Благодаря ему повторный запуск ячеек работает мгновенно и не нагружает сервер. Именно поэтому в папке модуля вы можете видеть директорию `cache` — это сохранённые ответы Overpass.


In [ ]:
ox.settings.use_cache = True
ox.settings.cache_folder = "./cache"

## 6. Смена геокодера Nominatim


Если ошибка возникает уже на шаге `ox.geocode_to_gdf()` (то есть при определении границ по названию), проблема не в Overpass, а в геокодере **Nominatim**. Его адрес меняется аналогично:

```python
# По умолчанию
ox.settings.nominatim_url = "https://nominatim.openstreetmap.org/"

# Зеркало (Германия)
ox.settings.nominatim_url = "https://nominatim.openstreetmap.de/"
```

> У публичного Nominatim строгая политика использования: не более **1 запроса в секунду** и обязательный заголовок с именем приложения. Для разовых учебных запросов сервера по умолчанию обычно достаточно; менять его стоит, только если он недоступен из вашей сети.


## 7. Проверка: пробная загрузка


Соберём все ключевые настройки в одну ячейку — её удобно ставить в начало любого ноутбука, где вы работаете с OSM:


In [5]:
ox.settings.overpass_url = "https://maps.mail.ru/osm/tools/overpass/api"  # зеркало Overpass
ox.settings.requests_timeout = 600                              # таймаут, сек
ox.settings.overpass_rate_limit = True                          # соблюдать лимиты сервера
ox.settings.use_cache = True                                    # кешировать ответы

Сделаем небольшой тестовый запрос — выгрузим кафе в пределах небольшой территории. Если ячейка отрабатывает без ошибок, настройки рабочие:


In [6]:
area_name = "Центральный район, Санкт-Петербург"
tags = {"amenity": "cafe"}

test_data = ox.features_from_place(area_name, tags)
len(test_data)

832

In [ ]:
test_data.explore(tiles='cartodbpositron')